In [45]:
import pandas as pd
import requests
import time
import os
import logging
from scrapy import Selector
from scrapy.crawler import CrawlerProcess
import scrapy
import boto3
from dotenv import load_dotenv
import numpy as np
from fake_useragent import UserAgent





In [ ]:
load_dotenv()
AWS_KEY = os.environ["AWS_KEY"]
AWS_SECRET_KEY = os.environ["AWS_SECRET_KEY"]
BUCKET = os.environ["BUCKET"]
API_Key = os.environ["OWM_Token"]

In [47]:
data_path = {
    "gps" : "data_gps_26.csv",
    "weather" : "data_weather_26.csv",
    "booking" : "data_booking_26.csv"
}

possible_destination = ["Mont Saint Michel",
"St Malo",
"Bayeux",
"Le Havre",
"Rouen",
"Paris",
"Amiens",
"Lille",
"Strasbourg",
"Chateau du Haut Koenigsbourg",
"Colmar",
"Eguisheim",
"Besancon",
"Dijon",
"Annecy",
"Grenoble",
"Lyon",
"Gorges du Verdon",
"Bormes les Mimosas",
"Cassis",
"Marseille",
"Aix en Provence",
"Avignon",
"Uzes",
"Nimes",
"Aigues Mortes",
"Saintes Maries de la mer",
"Collioure",
"Carcassonne",
"Ariege",
"Toulouse",
"Montauban",
"Biarritz",
"Bayonne",
"La Rochelle"]


# Weather API Data

In [48]:
#redefining agent

user_agent_list = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36"
]

user_agent = user_agent_list[0]

In [49]:
#Acquisition of coordinates of the cities looked into

gps_url =    "https://nominatim.openstreetmap.org/"
target = "search"

data_gps = {}

for city in possible_destination:
    gps_data = requests.get(gps_url+target, params={"q" : f"{city}, France", "format" : "json"}, headers= {"User-Agent" : user_agent})
    time.sleep(1.01)
    if len(gps_data.json()) > 0:
        data_city = gps_data.json()[0]
        data_gps[city] = {
            "lat" : data_city['lat'],
            "lon" : data_city['lon']
        }

    else:
        print(f"No data for {city}")
data_gps

{'Mont Saint Michel': {'lat': '48.6359541', 'lon': '-1.5114600'},
 'St Malo': {'lat': '48.6495180', 'lon': '-2.0260409'},
 'Bayeux': {'lat': '49.2764624', 'lon': '-0.7024738'},
 'Le Havre': {'lat': '49.4938975', 'lon': '0.1079732'},
 'Rouen': {'lat': '49.4404591', 'lon': '1.0939658'},
 'Paris': {'lat': '48.8534951', 'lon': '2.3483915'},
 'Amiens': {'lat': '49.8941708', 'lon': '2.2956951'},
 'Lille': {'lat': '50.6365654', 'lon': '3.0635282'},
 'Strasbourg': {'lat': '48.5846140', 'lon': '7.7507127'},
 'Chateau du Haut Koenigsbourg': {'lat': '48.2493820', 'lon': '7.3439412'},
 'Colmar': {'lat': '48.0777517', 'lon': '7.3579641'},
 'Eguisheim': {'lat': '48.0447968', 'lon': '7.3079618'},
 'Besancon': {'lat': '47.2380222', 'lon': '6.0243622'},
 'Dijon': {'lat': '47.3215806', 'lon': '5.0414701'},
 'Annecy': {'lat': '45.8992348', 'lon': '6.1288847'},
 'Grenoble': {'lat': '45.1875602', 'lon': '5.7357819'},
 'Lyon': {'lat': '45.7578137', 'lon': '4.8320114'},
 'Gorges du Verdon': {'lat': '43.74965

In [7]:
# gps_df = pd.DataFrame(city_gps).pivot_table()

#.to_csv(data_path["gps"])


In [ ]:
#From the GPS data, we can access the weather data to obtain the prediction for the next week.

params = {"units" : "metric",
          "exclude": "hourly, current, minutely, alerts"}

weather_ls = []
for key, value in data_gps.items():
    #time.sleep(1.02)
    url = f"https://api.openweathermap.org/data/2.5/forecast?lat={value['lat']}&lon={value['lon']}&appid={API_Key}"
    data = requests.get(url=url, params=params).json()["list"]
    for elem in data:
        if "rain" in elem.keys():
            rain = elem["rain"]["3h"]
        else:
            rain = 0

        weather_ls.append({
            "city" : key,
            "date" : pd.to_datetime(elem["dt"], unit='s', utc = True).date(),
            "cloudiness" : elem["clouds"]["all"],
            "temp_min": elem["main"]["temp_min"],
            "temp_max" : elem["main"]["temp_max"],
            "humidity" : elem["main"]["humidity"],
            "pop" : elem.get("pop", 0),
            "rain" : rain

        })


data_weather = pd.DataFrame(weather_ls)
data_weather


,city,date,cloudiness,temp_min,temp_max,humidity,pop,rain
0,Mont Saint Michel,2026-02-09,100,8.01,8.85,88,0.29,0.39
1,Mont Saint Michel,2026-02-09,100,8.57,9.06,92,1.00,4.05
2,Mont Saint Michel,2026-02-09,100,10.32,10.32,96,1.00,1.95
3,Mont Saint Michel,2026-02-09,94,9.13,9.13,96,0.76,0.42
4,Mont Saint Michel,2026-02-10,97,10.15,10.15,92,1.00,2.74
...,...,...,...,...,...,...,...,...
1395,La Rochelle,2026-02-13,46,7.67,7.67,83,0.45,0.26
1396,La Rochelle,2026-02-14,73,6.05,6.05,81,0.28,0.20
1397,La Rochelle,2026-02-14,100,5.23,5.23,73,0.20,0.12
1398,La Rochelle,2026-02-14,81,4.52,4.52,69,0.00,0.00


In [ ]:
data_gps_df = pd.DataFrame(data_gps)
mean_weather = data_weather.drop("date", axis=1).groupby("city").mean().sort_values("temp_max", ascending=False)
mean_weather = mean_weather.reset_index().merge(data_gps_df.transpose().reset_index().rename(columns={"index":"city"}), on="city", how="left", validate="1:1")
mean_weather["size"] = (mean_weather["temp_max"]-mean_weather["temp_max"].min())/(mean_weather["temp_max"].max()-mean_weather["temp_max"].min())
mean_weather["temp_mean"] = (mean_weather["temp_max"]+mean_weather["temp_max"])/2
best_weather = mean_weather.head(5)
best_weather

In [58]:
best_weather["city"]

0                    Biarritz
1                   Marseille
2                     Bayonne
3                   Collioure
4    Saintes Maries de la mer
Name: city, dtype: object

In [ ]:
data_weather.to_csv(data_path["weather"])

Scrapping of booking

necessary information:
- Nombre de nuits
- Nombre de personnes
- dates

In [12]:
#start_urls = "https://www.imdb.com/chart/boxoffice"
# base_url = "https://www.booking.com/searchresults.html?ss=Lyon"
base_url = "https://www.booking.com/hotel/fr/le-relais-saint-michel.fr.html?aid=356980&label=mkt123sc-9ae2f832-8959-47ab-9b0f-3440cd416970&sid=53174f83b56b159e3a4977053ec251e9&all_sr_blocks=23079515_93716278_0_1_0&checkin=2026-02-17&checkout=2026-02-19&dist=0&group_adults=2&group_children=0&hapos=5&highlighted_blocks=23079515_93716278_0_1_0&hpos=5&matching_block_id=23079515_93716278_0_1_0&no_rooms=1&req_adults=2&req_children=0&room1=A%2CA&sb_price_type=total&sr_order=popularity&sr_pri_blocks=23079515_93716278_0_1_0__49500&srepoch=1770131460&srpvid=ec3baffc890a6439cbfdf35d5b4a5633&type=total&ucfs=1&"




headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3 ',
    "Accept-Language": "fr-FR,fr;q=0.9"
}
url = base_url.split('.html')[0]
print(url)
r = requests.get(url, headers=headers)
print(r.status_code)
response = Selector(text=r.text)



https://www.booking.com/hotel/fr/le-relais-saint-michel.fr
202


In [29]:
base_url.split('?')[0]

'https://www.booking.com/hotel/fr/le-relais-saint-michel.fr.html'

form_booking = response.css("div[data-testid='property-card-container']")
name = form_booking.css("div[data-testid='title']::text").getall()
url = form_booking.css("a[data-testid='title-link']::attr(href)").getall()
score = form_booking.css("div[data-testid='review-score']")
close = score.css("div[aria-hidden='False']")
review_score = score.css("div[aria-hidden]::text").getall()
description = form_booking.css("div[class='fff1944c52']::text").getall()
loc = form_booking.css("span[data-testid='address']::text").getall()
#Location can be improved by looking at the hotel page directly. to be done later.



In [65]:
NOMINATIM_URL = "https://nominatim.openstreetmap.org/search"

HEADERS = {
    "User-Agent": "hotel-location-demo/1.0 (dubos.john@hotmail.fr)"
}

def get_hotel_location(hotel_name, city=None, country=None):
    """
    Returns best match location data for a hotel name using OpenStreetMap Nominatim.
    """

    query = hotel_name
    if country:
        query += f", {city}, {country}"

    params = {
        "q": query,
        "format": "json",
        "limit": 1,
        "addressdetails": 1
    }

    response = requests.get(
        NOMINATIM_URL,
        params=params,
        headers=HEADERS,
        timeout=10
    )
    response.raise_for_status()

    results = response.json()
    if not results:
        return None

    r = results[0]

    return {
        "name": hotel_name,
        "latitude": float(r["lat"]),
        "longitude": float(r["lon"]),
        "display_name": r["display_name"],
        "address": r.get("address", {})
    }



In [66]:
base_url = "https://www.booking.com/searchresults.html?"

user_agent = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:143.0) Gecko/20100101 Firefox/143.0 Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/140.0.0.0 Safari/537.36'
ua = UserAgent(platforms="desktop", min_version=120.0)
headers = {
    'User-Agent': user_agent,
    "Accept-Language": "fr-FR,fr;q=0.9"
}

hotel_city = {}
data_booking = []
response_ls = {}
r_ls = {}
visited = []
for city in best_weather["city"]:
    visited.append(city)
    params = {
        "ss" : city.replace(' ', '+'),
        "checkin" : "2026-02-17",
        "checkout" : "2026-02-19",
        "group_adults" : 2,
        "no_rooms" : 1,
        "group_children" : 0,
        "dest_type" : "city"
    }
    
    r = requests.get(base_url, params=params, headers= {
            'User-Agent' : ua.random, 
            "Accept-Language": "fr-FR,fr;q=0.9",
            "Connection": "keep-alive",
            "Upgrade-Insecure-Requests": "1",
            "Referer": "https://www.google.com/"})
    while r.status_code !=200:
        del r
        r = requests.get(base_url, params=params, headers= {
            'User-Agent' : ua.random, 
            "Accept-Language": 
            "fr-FR,fr;q=0.9",
            "Connection": "keep-alive",
            "Upgrade-Insecure-Requests": "1",
            "Referer": "https://www.google.com/"})
        time.sleep(1+np.random.rand())
    print(r.url)
    print(r.status_code)
    
    response = Selector(text=r.text)
    r_ls[city] = r
    response_ls[city] = response
    form_booking = response.css("div[data-testid='property-card']")
    for form in form_booking:
        name = form.css("div[data-testid='title']::text").get().split(',')[0].split('-')[0]
        url = form.css("a[data-testid='title-link']::attr(href)").get()
        score = form.css("div[data-testid='review-score']")
        close = score.css("div[aria-hidden='False']")
        review_score = score.css("div[aria-hidden]::text").get()
        description = form.css("div[class='fff1944c52']::text").get()

        hotel_info = get_hotel_location(name, city=city, country="France")
        time.sleep(1)


        # r_loc = requests.get(url=url.split('?')[0], headers={
        #     'User-Agent' : ua.random, 
        #     "Accept-Language": 
        #     "fr-FR,fr;q=0.9",
        #     "Connection": "keep-alive",
        #     "Upgrade-Insecure-Requests": "1",
        #     "Referer": "https://www.google.com/"
        # })
        # print(r_loc.url)
        # print(r_loc.status_code)
        # time.sleep(1+np.random.rand())
        # response_loc = Selector(text=r_loc.text)
        # content = response_loc.css("div[data-testid='map-entry-point-desktop-wrapper']::attr(data-atlas-latlng)").get()
        # lat = None
        # lon = None
        # if content:
        #     lat, lon = content.split(',')
        # loc = form.css("span[data-testid='address']::text").get()
        # description = [txt for txt in description if "Hébergement géré par un particulier" not in txt]

        if hotel_info is not None:
            data_booking.append({
                "city" : city,
                "name" : name,
                "url" : url,
                "review_score" : review_score,
                "description" : description,
                "location" : hotel_info["address"],
                "lat": hotel_info["latitude"],
                "lon":hotel_info["longitude"]
            })
        else:
            data_booking.append({
                "city" : city,
                "name" : name,
                "url" : url,
                "review_score" : review_score,
                "description" : description,
                "location" : "missing value",
                "lat": None,
                "lon": None
            })


hotel_df = (pd.DataFrame(data_booking))

https://www.booking.com/searchresults.html?ss=Biarritz&checkin=2026-02-17&checkout=2026-02-19&group_adults=2&no_rooms=1&group_children=0&dest_type=city
200
https://www.booking.com/searchresults.html?ss=Marseille&checkin=2026-02-17&checkout=2026-02-19&group_adults=2&no_rooms=1&group_children=0&dest_type=city
200
https://www.booking.com/searchresults.html?ss=Bayonne&checkin=2026-02-17&checkout=2026-02-19&group_adults=2&no_rooms=1&group_children=0&dest_type=city
200
https://www.booking.com/searchresults.html?ss=Collioure&checkin=2026-02-17&checkout=2026-02-19&group_adults=2&no_rooms=1&group_children=0&dest_type=city
200
https://www.booking.com/searchresults.html?ss=Saintes%2BMaries%2Bde%2Bla%2Bmer&checkin=2026-02-17&checkout=2026-02-19&group_adults=2&no_rooms=1&group_children=0&dest_type=city
200


In [62]:
len(visited)

5

In [68]:
hotel_df = pd.DataFrame(data_booking)
hotel_df.to_csv(data_path["booking"])
hotel_df

,city,name,url,review_score,description,location,lat,lon
0,Biarritz,Hôtel Barnea,https://www.booking.com/hotel/fr/hotelargieder...,"8,9",1 grand lit double,missing value,NaN,NaN
1,Biarritz,Le Talaia Hôtel & Spa Biarritz,https://www.booking.com/hotel/fr/radisson-blu-...,"8,8",1 lit simple,"{'tourism': 'Le Talaia Hôtel & Spa Biarritz', ...",43.478210,-1.564422
2,Biarritz,Hôtel Kemaris,https://www.booking.com/hotel/fr/best-western-...,"8,4",1 grand lit double,"{'tourism': 'Hôtel Best Western Kemaris', 'roa...",43.476518,-1.562496
3,Biarritz,ALFRED HOTELS Les Halles,https://www.booking.com/hotel/fr/hotelanjou.fr...,"9,1",1 grand lit double,missing value,NaN,NaN
4,Biarritz,Hôtel du Palais Biarritz,https://www.booking.com/hotel/fr/hoteldupalais...,"9,5",2 lits simples,"{'tourism': 'Hôtel du Palais', 'road': 'Avenue...",43.486435,-1.556328
...,...,...,...,...,...,...,...,...
122,Saintes Maries de la mer,Hôtel Le Neptune en Camargue,https://www.booking.com/hotel/fr/le-neptune-en...,"8,4",Hébergement géré par un particulier,missing value,NaN,NaN
123,Saintes Maries de la mer,Le Sauvage,https://www.booking.com/hotel/fr/le-sauvage-le...,"9,3",Hébergement géré par un particulier,missing value,NaN,NaN
124,Saintes Maries de la mer,De l'eau et des oiseaux !,https://www.booking.com/hotel/fr/de-l-eau-et-d...,"8,8",Hébergement géré par un particulier,missing value,NaN,NaN
125,Saintes Maries de la mer,Hôtel Spa Mas des Rièges,https://www.booking.com/hotel/fr/mas-des-riege...,"9,0","2 lits (1 canapé-lit, 1 grand lit double)",missing value,NaN,NaN


In [ ]:
class BookingSpider(scrapy.Spider):
    name = "Booking_data"
    start_urls = ["https://www.booking.com/"]

    def parse(response):
        hotels = response.css("div[data-testid='title']")
        for hotel in hotels:
            yield {
                "hotel_name" : hotel.css("div::text")
            }
        return response



In [51]:
!scrapy crawl Booking_data

test
[<Selector query="descendant-or-self::div[@data-testid = 'property-card-container']" data='<div class="aa97d6032f" data-testid="...'>, <Selector query="descendant-or-self::div[@data-testid = 'property-card-container']" data='<div class="aa97d6032f" data-testid="...'>, <Selector query="descendant-or-self::div[@data-testid = 'property-card-container']" data='<div class="aa97d6032f" data-testid="...'>, <Selector query="descendant-or-self::div[@data-testid = 'property-card-container']" data='<div class="aa97d6032f" data-testid="...'>, <Selector query="descendant-or-self::div[@data-testid = 'property-card-container']" data='<div class="aa97d6032f" data-testid="...'>, <Selector query="descendant-or-self::div[@data-testid = 'property-card-container']" data='<div class="aa97d6032f" data-testid="...'>, <Selector query="descendant-or-self::div[@data-testid = 'property-card-container']" data='<div class="aa97d6032f" data-testid="...'>, <Selector query="descendant-or-self::div[@data-testid = 

2025-09-12 18:05:25 [scrapy.utils.log] INFO: Scrapy 2.13.3 started (bot: tutorial)
2025-09-12 18:05:26 [scrapy.utils.log] INFO: Versions:
{'lxml': '5.3.0',
 'libxml2': '2.13.8',
 'cssselect': '1.2.0',
 'parsel': '1.8.1',
 'w3lib': '2.1.2',
 'Twisted': '24.11.0',
 'Python': '3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, '
           '16:37:03) [MSC v.1929 64 bit (AMD64)]',
 'pyOpenSSL': '25.1.0 (OpenSSL 3.0.17 1 Jul 2025)',
 'cryptography': '45.0.5',
 'Platform': 'Windows-11-10.0.26100-SP0'}
2025-09-12 18:05:26 [scrapy.addons] INFO: Enabled addons:
[]
2025-09-12 18:05:26 [asyncio] DEBUG: Using selector: SelectSelector
2025-09-12 18:05:26 [scrapy.utils.log] DEBUG: Using reactor: twisted.internet.asyncioreactor.AsyncioSelectorReactor
2025-09-12 18:05:26 [scrapy.utils.log] DEBUG: Using asyncio event loop: asyncio.windows_events._WindowsSelectorEventLoop
2025-09-12 18:05:26 [scrapy.extensions.telnet] INFO: Telnet Password: bee971521fe2b05d
2025-09-12 18:05:26 [scrapy.middleware]

Loading all data on the datalake (S3)

In [46]:
session = boto3.Session(aws_access_key_id = AWS_KEY, 
                        aws_secret_access_key = AWS_SECRET_KEY, 
                        region_name = "eu-west-3")

s3 = session.resource("s3")


In [52]:
my_bucket = s3.Bucket(BUCKET)
for path in data_path.values():
    my_bucket.upload_file(path, f"S3_{path}")